# 06 — Génération de `data/upsert/demographics.csv`

Table de calibration OFS par tranche d'âge du simulateur, **données pré-Covid** :

| Colonne(s) | Source OFS | Fichier `data/raw/` |
|---|---|---|
| `median_monthly_gross_chf` | ESS — salaire brut médian | `px-x-0304010000_205.px` |
| `concert_participation_ratio_pct`, `annual_concert_quantity` | Statistique des pratiques culturelles **2019** (concert / spectacle musical) | `participationDemographics.csv` |
| `base_budget_per_concert_chf`, `dr_min/max_budget_chf` | Enquête sur le budget des ménages **2018-2019** (dépense « théâtre et concerts ») | `budgetRatioDemographics.csv` |

Chaque fichier `data/raw/` est réduit aux seules **tranches d'âge** utiles (5 classes
OFS pour les concerts, âge de la personne de référence du ménage pour le budget),
remappées sur les 5 tranches du simulateur définies dans `env/refdata.py`.

> `refdata.py` ne consomme que `median_monthly_gross_chf`,
> `concert_participation_ratio_pct` et `annual_concert_quantity` (le budget-billet
> effectif y est dérivé du **revenu discrétionnaire**). Les colonnes
> `*_budget_chf` restent indicatives / documentaires.

In [1]:
%pip install pyaxis
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import re

import numpy as np
import pandas as pd
from pyaxis import pyaxis

RAW = "../data/raw"
OUT = "../data/upsert/demographics.csv"

# Tranches d'âge du simulateur — cf. env/refdata.py (AGE_BRACKETS)
BRACKETS = ["13-17", "18-24", "25-34", "35-44", "45_plus"]

In [3]:
def _num(x):
    """'78,9' -> 78.9 ; '15,46 c' -> 15.46 ; '', '()', '*' -> NaN."""
    s = str(x).strip().replace(",", ".")
    m = re.search(r"-?\d+\.?\d*", s)
    return float(m.group()) if m else np.nan


# --------------------------------------------------------------------------- #
# 1. Salaires : OFS ESS, px-x-0304010000_205 (structure = enquête, non Covid)  #
# --------------------------------------------------------------------------- #
def median_salaries(px_path=f"{RAW}/px-x-0304010000_205.px"):
    """Salaire mensuel brut médian (CHF), Suisse, ensemble des secteurs.

    Les 13-17 / 18-24 ne travaillent pas à plein temps : fraction du médian des
    < 30 ans (hypothèse de modélisation, analyse de sensibilité dans le rapport).
    35-44 -> médian 30-49 ans ; 45_plus -> médian 50 ans et plus.
    """
    df_px = pyaxis.parse(px_path, encoding="latin-1")["DATA"]
    cols = list(df_px.columns)
    year_col = [c for c in cols if df_px[c].astype(str).str.contains("2024|2022", na=False).any()][0]
    region_col = [c for c in cols if df_px[c].astype(str).str.contains("Schweiz|Suisse", na=False).any()][0]
    total_col = [c for c in cols if df_px[c].astype(str).str.contains("Total|total", na=False).any()][0]
    age_col = [c for c in cols if df_px[c].astype(str).str.contains("29", na=False).any()][0]
    metric_col = [c for c in cols if df_px[c].astype(str).str.contains("Zentralwert|Médiane", na=False).any()][0]

    df_sal = df_px[
        df_px[region_col].astype(str).str.contains("Schweiz|Suisse", na=False)
        & df_px[total_col].astype(str).str.contains("Total|total", na=False)
        & df_px[metric_col].astype(str).str.contains("Zentralwert|Médiane", na=False)
        & ~df_px[age_col].astype(str).str.contains("Total|total", na=False)
    ].copy()
    df_sal = df_sal[df_sal[year_col] == df_sal[year_col].max()]

    def med(pat):
        m = df_sal[df_sal[age_col].astype(str).str.contains(pat, regex=True, na=False)]
        return float(m["DATA"].iloc[0])

    young = med("29")
    return {
        "13-17": young * 0.30,
        "18-24": young * 0.80,
        "25-34": young,
        "35-44": med("30.*49"),
        "45_plus": med("50"),
    }


# --------------------------------------------------------------------------- #
# 2. Concerts : OFS Statistique des pratiques culturelles 2019 (pré-Covid)     #
#    participationDemographics.csv — 5 classes d'âge, part + bandes de fréq.    #
# --------------------------------------------------------------------------- #
def participation_2019(path=f"{RAW}/participationDemographics.csv"):
    """Par classe d'âge OFS : part ayant fréquenté un concert / spectacle musical
    dans les 12 mois (%), et nombre moyen de sorties/an parmi ces personnes
    (milieux de tranche : « 1-3 fois » -> 2, « 4-6 » -> 5, « 7 et plus » -> 7.5).
    """
    df = pd.read_csv(path, sep=";", dtype=str).fillna("")
    out = {}
    for _, r in df.iterrows():
        part = _num(r["Fréquenté"])
        occ = _num(r["1-3 fois (occasionnellement)"])
        reg = _num(r["4-6 fois (régulièrement)"])
        ass = _num(r["7 fois et plus (assidûment)"])
        freq = (occ * 2.0 + reg * 5.0 + ass * 7.5) / part
        out[r["Age"].strip()] = {"part": part, "freq": freq}
    return out


# --------------------------------------------------------------------------- #
# 3. Budget : OFS Enquête sur le budget des ménages 2018-2019 (pré-Covid)      #
#    budgetRatioDemographics.csv — CHF/mois/ménage « théâtre et concerts »,    #
#    par classe d'âge de la personne de référence                             #
# --------------------------------------------------------------------------- #
def theatre_concert_spend(path=f"{RAW}/budgetRatioDemographics.csv"):
    df = pd.read_csv(path, sep=";", dtype=str).fillna("")
    row = df[df.iloc[:, 0].str.contains("Théâtre et concerts", na=False)].iloc[0]
    return {
        "<=34": _num(row["Jusqu'à 34 ans (Montant)"]),
        "35-44": _num(row["35 - 44 ans (Montant)"]),
        "45-54": _num(row["45 - 54 ans (Montant)"]),
        "55-64": _num(row["55 - 64 ans (Montant)"]),
        "65-74": _num(row["65 - 74 ans (Montant)"]),
        ">=75": _num(row["Dès 75 ans (Montant)"]),
    }


# --------------------------------------------------------------------------- #
# 4. Assemblage : classes d'âge OFS -> tranches du simulateur                  #
# --------------------------------------------------------------------------- #
def build_demographics():
    sal = median_salaries()
    p = participation_2019()
    tc = theatre_concert_spend()

    # 13-17 : la population < 15 ans n'est pas enquêtée. On part de la classe
    # 15-29 ans, ramenée au profil des 15-19 ans observé en 2024 : ratio
    # 52.2 / 73.8 = 0.71 (appliqué à la participation seulement).
    teen = 52.2 / 73.8

    def blend(field, teen_scale):
        return {
            "13-17": p["15-29 ans"][field] * teen_scale,
            "18-24": p["15-29 ans"][field],
            "25-34": (p["15-29 ans"][field] + p["30-44 ans"][field]) / 2,
            "35-44": p["30-44 ans"][field],
            "45_plus": (
                0.45 * p["45-59 ans"][field]
                + 0.35 * p["60-74 ans"][field]
                + 0.20 * p["75 ans et plus"][field]
            ),
        }

    part = blend("part", teen)
    freq = blend("freq", 1.0)
    spend = {
        "13-17": tc["<=34"],
        "18-24": tc["<=34"],
        "25-34": tc["<=34"],
        "35-44": tc["35-44"],
        "45_plus": 0.30 * tc["45-54"] + 0.28 * tc["55-64"] + 0.24 * tc["65-74"] + 0.18 * tc[">=75"],
    }

    rows = []
    for b in BRACKETS:
        vol = 0.40 if b in ("13-17", "18-24") else 0.25
        qty = round(freq[b], 1)
        # budget-billet indicatif : 50 % du budget annuel théâtre/concerts / sorties par an
        base = spend[b] * 12 * 0.50 / qty
        rows.append(
            {
                "age_bracket": b,
                "median_monthly_gross_chf": round(sal[b], 1),
                "concert_participation_ratio_pct": round(part[b], 1),
                "annual_concert_quantity": qty,
                "base_budget_per_concert_chf": round(base, 2),
                "budget_volatility_pct": vol,
                "dr_min_budget_chf": round(base * (1 - vol), 2),
                "dr_max_budget_chf": round(base * (1 + vol), 2),
            }
        )

    os.makedirs(os.path.dirname(OUT), exist_ok=True)
    df = pd.DataFrame(rows)
    df.to_csv(OUT, index=False, encoding="utf-8-sig")
    return df

In [4]:
df_demographics = build_demographics()
print(f"OK — {OUT} régénéré (OFS : pratiques culturelles 2019 + budget des ménages 2018-2019)")
df_demographics

Multilingual PX file


OK — ../data/upsert/demographics.csv régénéré (OFS : pratiques culturelles 2019 + budget des ménages 2018-2019)


,age_bracket,median_monthly_gross_chf,concert_participation_ratio_pct,annual_concert_quantity,base_budget_per_concert_chf,budget_volatility_pct,dr_min_budget_chf,dr_max_budget_chf
0,13-17,1671.0,55.8,4.1,16.73,0.40,10.04,23.42
1,18-24,4456.0,78.9,4.1,16.73,0.40,10.04,23.42
2,25-34,5570.0,77.5,4.0,17.14,0.25,12.86,21.43
3,35-44,7288.0,76.1,3.8,19.48,0.25,14.61,24.36
4,45_plus,7806.0,67.9,3.9,26.48,0.25,19.86,33.10
